# Notebook 04 — Master Player Table

This notebook merges all three data sources into a single master table keyed on `mlbam_id`.

**Merge strategy:**
- Join key: `mlbam_id` (universal across all sources)
- FanGraphs data is joined via the `fg_mlbam_crosswalk` bridge table
- Statcast data already uses `mlbam_id` directly

Stat conflict hierarchy:
| Stat type | Preferred source | Reason |
|---|---|---|
| Rate stats (ERA, wOBA, FIP, etc.) | FanGraphs | Park-adjusted, more methodologically rigorous |
| Counting stats (IP, PA, HR, etc.) | MLB Stats API | Official source of record |
| Statcast metrics (xwOBA, xERA, barrel%) | Baseball Savant | Exclusive to Statcast |

Minimum thresholds: PA ≥ 50 for hitters, IP ≥ 10 for pitchers  
Confidence flag: `low_confidence = True` if player appears in only 1 of 3 sources

In [32]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA = os.path.join("..", "data")

## 1. Load all raw data

In [34]:
# MLB Stats API
mlb_players = pd.read_parquet(os.path.join(DATA, "raw_mlb_api_players.parquet"))
mlb_hitting = pd.read_parquet(os.path.join(DATA, "raw_mlb_api_hitting.parquet"))
mlb_pitching = pd.read_parquet(os.path.join(DATA, "raw_mlb_api_pitching.parquet"))

# Statcast 
sc_hitting = pd.read_parquet(os.path.join(DATA, "raw_statcast_hitting.parquet"))
sc_pitching = pd.read_parquet(os.path.join(DATA, "raw_statcast_pitching.parquet"))

# FanGraphs
fg_hitting = pd.read_parquet(os.path.join(DATA, "raw_fangraphs_hitting.parquet"))
fg_pitching = pd.read_parquet(os.path.join(DATA, "raw_fangraphs_pitching.parquet"))
park_factors = pd.read_parquet(os.path.join(DATA, "park_factors.parquet"))
crosswalk = pd.read_parquet(os.path.join(DATA, "fg_mlbam_crosswalk.parquet"))

print("Raw data loaded:")
for name, df in [("mlb_players", mlb_players), ("mlb_hitting", mlb_hitting),
                  ("mlb_pitching", mlb_pitching), ("sc_hitting", sc_hitting),
                  ("sc_pitching", sc_pitching), ("fg_hitting", fg_hitting),
                  ("fg_pitching", fg_pitching), ("park_factors", park_factors),
                  ("crosswalk", crosswalk)]:
    print(f"  {name:20s} {df.shape}")

Raw data loaded:
  mlb_players          (1470, 4)
  mlb_hitting          (673, 13)
  mlb_pitching         (802, 13)
  sc_hitting           (537, 23)
  sc_pitching          (657, 18)
  fg_hitting           (537, 10)
  fg_pitching          (657, 10)
  park_factors         (30, 3)
  crosswalk            (1193, 5)


## 2. Prepare FanGraphs data with MLBAM IDs

Attach `mlbam_id` to FanGraphs DataFrames via the crosswalk, then select only the columns we want from each source (following the conflict hierarchy).

In [36]:
#build crosswalk lookup: fg_id -> mlbam_id
xwalk = crosswalk[crosswalk["match_status"] == "matched"][["fg_id", "mlbam_id"]].copy()
xwalk["mlbam_id"] = xwalk["mlbam_id"].astype(int)

#attach mlbam_id to FanGraphs hitting
fg_hit = fg_hitting.merge(xwalk, on="fg_id", how="inner")
#rate stats from FanGraphs: wOBA, BABIP, wRC+, WAR, batted ball profile
fg_hit_cols = fg_hit[["mlbam_id", "wOBA", "BABIP", "wRC_plus", "WAR",
                       "GB_pct", "FB_pct", "LD_pct"]].copy()
fg_hit_cols = fg_hit_cols.rename(columns={"WAR": "fg_WAR_bat"})

#attach mlbam_id to FanGraphs pitching
fg_pit = fg_pitching.merge(xwalk, on="fg_id", how="inner")
#rate stats from FanGraphs: FIP, xFIP, SIERA, WAR, GB%, K%, BB%
fg_pit_cols = fg_pit[["mlbam_id", "FIP", "xFIP", "SIERA",
                       "WAR", "GB_pct", "K_pct", "BB_pct"]].copy()
fg_pit_cols = fg_pit_cols.rename(columns={
    "WAR": "fg_WAR_pit",
    "GB_pct": "fg_GB_pct",
    "K_pct": "fg_K_pct",
    "BB_pct": "fg_BB_pct",
})

print(f"FG hitting with mlbam_id: {len(fg_hit_cols)} rows")
print(f"FG pitching with mlbam_id: {len(fg_pit_cols)} rows")

FG hitting with mlbam_id: 537 rows
FG pitching with mlbam_id: 653 rows


## 3. Prepare Statcast data

Select only the Statcast-exclusive metrics.

In [38]:
# Statcast hitting
sc_hit_cols = sc_hitting[["mlbam_id", "xBA", "xSLG", "xwOBA", "xOBP", "xISO", "avg_swing_speed", "fast_swing_rate", "blasts_contact",
                          "blasts_swing", "squared_up_contact", "squared_up_swing", "avg_swing_length", "swords", "attack_angle",
                          "attack_direction", "ideal_angle_rate", "vertical_swing_path", "exit_velocity_avg", "launch_angle_avg",
                          "sweet_spot_percent", "barrel_batted_rate"]].copy()

# Statcast pitching
sc_pit_cols = sc_pitching[["mlbam_id", "xERA", "xBA", "xSLG", "xwOBA", "xOBP", "xISO", "wOBA", "k_percent", "bb_percent", "whiff_percent",
                           "chase_percent", "exit_velocity_avg", "sweet_spot_percent", "barrel_batted_rate", "hard_hit_percent", 
                           "groundballs_percent"]].copy()

print(f"Statcast hitting: {len(sc_hit_cols)} rows")
print(f"Statcast pitching: {len(sc_pit_cols)} rows")

Statcast hitting: 537 rows
Statcast pitching: 657 rows


## 4. Prepare MLB Stats API data

Counting stats from the API. Convert string rate columns (AVG, OBP, SLG, OPS, ERA, WHIP) to float.

In [40]:
# MLB API hitting 
mlb_hit = mlb_hitting.copy()
for col in ["AVG", "OBP", "SLG", "OPS"]:
    mlb_hit[col] = pd.to_numeric(mlb_hit[col], errors="coerce")

# MLB API pitching
mlb_pit = mlb_pitching.copy()
for col in ["ERA", "WHIP"]:
    mlb_pit[col] = pd.to_numeric(mlb_pit[col], errors="coerce")

print(f"MLB API hitting: {len(mlb_hit)} rows")
print(f"MLB API pitching: {len(mlb_pit)} rows")

MLB API hitting: 673 rows
MLB API pitching: 802 rows


## 5. Build master hitter table

Merge strategy: Start with MLB API (the base), left-join FanGraphs and Statcast. Track which sources each player has data from.

In [42]:
# Start with MLB API as base (counting stats)
master_hit = mlb_hit.copy()
print(f"Step 0 — MLB API base:    {len(master_hit)} hitters")

# Join player info (name, position, team)
master_hit = master_hit.merge(
    mlb_players[["mlbam_id", "name", "position", "team"]],
    on="mlbam_id", how="left"
)

# Use primary_team (team with most PA) over currentTeam for traded players
if "primary_team" in master_hit.columns:
    master_hit["team"] = master_hit["primary_team"].fillna(master_hit["team"])
    master_hit = master_hit.drop(columns=["primary_team"])
    print("  → Using primary_team (most PA) for team assignment")

master_hit["in_mlb_api"] = True

# Join FanGraphs
master_hit = master_hit.merge(fg_hit_cols, on="mlbam_id", how="left")
master_hit["in_fangraphs"] = master_hit["wOBA"].notna()
print(f"Step 1 — After FG join:   {len(master_hit)} hitters ({master_hit['in_fangraphs'].sum()} have FG data)")

# Join Statcast
master_hit = master_hit.merge(sc_hit_cols, on="mlbam_id", how="left")
master_hit["in_statcast"] = master_hit["xwOBA"].notna()
print(f"Step 2 — After SC join:   {len(master_hit)} hitters ({master_hit['in_statcast'].sum()} have SC data)")

# Apply minimum threshold: PA >= 50
before_filter = len(master_hit)
master_hit = master_hit[master_hit["PA"] >= 50].reset_index(drop=True)
print(f"Step 3 — After PA >= 50:  {len(master_hit)} hitters (dropped {before_filter - len(master_hit)})")

master_hit["player_type"] = "hitter"

master_hit = master_hit.drop(columns=["in_mlb_api", "in_fangraphs", "in_statcast"])

print(f"\nFinal hitter table: {master_hit.shape}")
master_hit.head(10)

Step 0 — MLB API base:    673 hitters
  → Using primary_team (most PA) for team assignment
Step 1 — After FG join:   673 hitters (537 have FG data)
Step 2 — After SC join:   673 hitters (537 have SC data)
Step 3 — After PA >= 50:  537 hitters (dropped 136)

Final hitter table: (537, 44)


,mlbam_id,PA,AVG,OBP,SLG,OPS,HR,RBI,BB,SO,BB_pct,K_pct,name,position,team,wOBA,BABIP,wRC_plus,fg_WAR_bat,GB_pct,FB_pct,LD_pct,xBA,xSLG,xwOBA,xOBP,xISO,avg_swing_speed,fast_swing_rate,blasts_contact,blasts_swing,squared_up_contact,squared_up_swing,avg_swing_length,swords,attack_angle,attack_direction,ideal_angle_rate,vertical_swing_path,exit_velocity_avg,launch_angle_avg,sweet_spot_percent,barrel_batted_rate,player_type
0,596019,732,0.267,0.346,0.466,0.812,31,86,65,131,8.9,17.9,Francisco Lindor,SS,New York Mets,0.350,0.288,129.0,6.3,0.377,0.402,0.221,0.260,0.454,0.345,0.342,0.194,71.0,16.4,13.9,11.1,33.8,26.9,7.5,14.0,11.1,-1.6,51.1,33.8,90.5,15.1,34.6,8.8,hitter
1,646240,729,0.252,0.372,0.479,0.851,35,109,112,192,15.4,26.3,Rafael Devers,DH,San Francisco Giants,0.365,0.307,135.0,3.3,0.427,0.401,0.172,0.244,0.487,0.367,0.368,0.242,71.6,19.8,17.9,12.3,31.5,21.7,7.6,28.0,10.0,4.5,59.5,29.9,93.5,12.6,34.6,16.0,hitter
2,660271,727,0.282,0.392,0.622,1.014,55,102,109,187,15.0,25.7,Shohei Ohtani,TWP,Los Angeles Dodgers,0.418,0.315,172.0,7.5,0.392,0.439,0.169,0.274,0.649,0.425,0.387,0.375,75.8,61.9,26.0,17.8,36.4,25.0,7.9,14.0,15.0,-2.0,53.3,36.9,94.9,15.0,35.9,23.5,hitter
3,656941,724,0.240,0.365,0.563,0.928,56,132,108,197,14.9,27.2,Kyle Schwarber,LF,Philadelphia Phillies,0.391,0.253,152.0,4.9,0.341,0.480,0.179,0.251,0.584,0.403,0.374,0.333,77.3,77.1,24.7,17.5,32.2,22.8,7.5,51.0,14.6,-5.8,64.3,30.3,94.3,20.1,35.3,20.8,hitter
4,621566,724,0.272,0.366,0.484,0.850,29,95,91,176,12.6,24.3,Matt Olson,1B,Atlanta Braves,0.366,0.333,136.0,4.7,0.395,0.400,0.205,0.249,0.492,0.360,0.347,0.244,73.9,41.3,18.1,13.5,29.3,21.9,7.5,21.0,8.7,1.1,48.2,33.9,93.3,14.8,37.1,14.3,hitter
5,672695,720,0.290,0.389,0.462,0.851,20,100,94,83,13.1,11.5,Geraldo Perdomo,SS,Arizona Diamondbacks,0.370,0.303,138.0,7.1,0.375,0.392,0.233,0.278,0.424,0.356,0.381,0.146,68.3,4.6,10.2,9.1,36.0,32.3,6.9,21.0,7.4,-1.8,49.9,35.4,87.6,15.5,36.2,6.2,hitter
6,665742,715,0.263,0.396,0.525,0.921,43,105,127,137,17.8,19.2,Juan Soto,RF,New York Mets,0.390,0.270,156.0,5.8,0.447,0.386,0.167,0.288,0.608,0.429,0.417,0.321,73.6,40.9,26.3,21.6,39.6,32.5,7.0,36.0,10.9,3.8,65.9,28.1,93.8,12.0,32.9,18.1,hitter
7,677594,710,0.267,0.324,0.474,0.798,32,95,44,152,6.2,21.4,Julio Rodríguez,CF,Seattle Mariners,0.341,0.302,126.0,5.7,0.472,0.345,0.183,0.274,0.483,0.348,0.331,0.209,76.4,63.9,19.8,14.4,30.8,22.4,7.7,19.0,6.9,-1.4,46.4,31.7,91.8,8.5,31.5,9.8,hitter
8,668227,709,0.238,0.334,0.426,0.760,27,76,64,191,9.0,26.9,Randy Arozarena,LF,Seattle Mariners,0.332,0.298,120.0,2.9,0.425,0.406,0.169,0.227,0.417,0.327,0.326,0.190,72.3,29.8,18.3,13.3,35.0,25.4,7.4,20.0,8.7,0.4,62.1,25.3,91.3,12.9,32.6,11.5,hitter
9,624413,709,0.272,0.347,0.524,0.871,38,126,61,162,8.6,22.8,Pete Alonso,1B,New York Mets,0.368,0.305,141.0,3.6,0.380,0.418,0.202,0.274,0.572,0.386,0.349,0.299,75.3,53.4,22.7,17.2,33.6,25.4,7.1,15.0,10.1,1.1,59.8,33.8,93.5,15.4,38.4,18.9,hitter


## 6. Build master pitcher table

Same merge strategy: MLB API base → left-join FanGraphs → left-join Statcast.

In [44]:
# Start with MLB API pitching as base
master_pit = mlb_pit.copy()
print(f"Step 0 — MLB API base:    {len(master_pit)} pitchers")

# Join player info
master_pit = master_pit.merge(
    mlb_players[["mlbam_id", "name", "position", "team"]],
    on="mlbam_id", how="left"
)

# Use primary_team (team with most IP) over currentTeam for traded players
if "primary_team" in master_pit.columns:
    master_pit["team"] = master_pit["primary_team"].fillna(master_pit["team"])
    master_pit = master_pit.drop(columns=["primary_team"])
    print("  → Using primary_team (most IP) for team assignment")

master_pit["in_mlb_api"] = True

# Join FanGraphs pitching 
master_pit = master_pit.merge(fg_pit_cols, on="mlbam_id", how="left")
master_pit["in_fangraphs"] = master_pit["FIP"].notna()
print(f"Step 1 — After FG join:   {len(master_pit)} pitchers ({master_pit['in_fangraphs'].sum()} have FG data)")

# Join Statcast pitching 
master_pit = master_pit.merge(sc_pit_cols, on="mlbam_id", how="left")
master_pit["in_statcast"] = master_pit["xERA"].notna()
print(f"Step 2 — After SC join:   {len(master_pit)} pitchers ({master_pit['in_statcast'].sum()} have SC data)")

# Apply minimum threshold: IP >= 10
before_filter = len(master_pit)
master_pit = master_pit[master_pit["IP"] >= 10].reset_index(drop=True)
print(f"Step 3 — After IP >= 10:  {len(master_pit)} pitchers (dropped {before_filter - len(master_pit)})")

master_pit["player_type"] = "pitcher"
master_pit = master_pit.drop(columns=["in_mlb_api", "in_fangraphs", "in_statcast"])

print(f"\nFinal pitcher table: {master_pit.shape}")
master_pit.head(10)

Step 0 — MLB API base:    802 pitchers
  → Using primary_team (most IP) for team assignment
Step 1 — After FG join:   802 pitchers (652 have FG data)
Step 2 — After SC join:   802 pitchers (656 have SC data)
Step 3 — After IP >= 10:  656 pitchers (dropped 146)

Final pitcher table: (656, 39)


,mlbam_id,GS,G,IP,ERA,WHIP,SO,BB,HR,K_per9,BB_per9,HR_per9,name,position,team,FIP,xFIP,SIERA,fg_WAR_pit,fg_GB_pct,fg_K_pct,fg_BB_pct,xERA,xBA,xSLG,xwOBA,xOBP,xISO,wOBA,k_percent,bb_percent,whiff_percent,chase_percent,exit_velocity_avg,sweet_spot_percent,barrel_batted_rate,hard_hit_percent,groundballs_percent,player_type
0,657277,34,34,207.0,3.22,1.24,224,46,14,9.74,2.00,0.61,Logan Webb,P,San Francisco Giants,2.60,2.78,3.14,5.5,0.532,0.262,0.054,3.58,0.246,0.385,0.295,0.292,0.138,0.303,26.2,5.4,24.7,29.7,89.7,32.8,8.3,40.2,54.2,pitcher
1,676979,32,32,205.1,2.59,1.03,255,46,24,11.19,2.02,1.05,Garrett Crochet,P,Boston Red Sox,2.89,2.64,2.86,5.8,0.483,0.313,0.057,2.89,0.210,0.347,0.266,0.258,0.137,0.270,31.3,5.7,29.4,32.6,87.7,32.2,7.3,37.3,49.3,pitcher
2,650911,32,32,202.0,2.50,1.06,212,44,12,9.45,1.96,0.53,Cristopher Sánchez,P,Philadelphia Phillies,2.55,2.77,3.02,6.4,0.583,0.263,0.055,3.02,0.225,0.345,0.272,0.273,0.120,0.264,26.3,5.5,30.4,31.6,89.0,27.0,5.7,39.8,58.5,pitcher
3,669373,31,31,195.1,2.21,0.89,241,33,18,11.12,1.52,0.83,Tarik Skubal,P,Detroit Tigers,2.45,2.66,2.71,6.6,0.410,0.322,0.044,2.71,0.206,0.344,0.258,0.246,0.138,0.245,32.2,4.4,32.5,35.1,86.1,35.6,8.1,33.0,41.6,pitcher
4,608331,32,32,195.1,2.86,1.10,189,51,14,8.72,2.35,0.65,Max Fried,P,New York Yankees,3.07,3.41,3.60,4.8,0.524,0.236,0.064,3.38,0.233,0.358,0.287,0.291,0.125,0.272,23.6,6.4,26.6,27.6,87.2,29.6,6.9,37.2,52.6,pitcher
5,607074,33,33,195.1,3.09,1.05,203,73,22,9.36,3.37,1.01,Carlos Rodón,P,New York Yankees,3.78,3.89,3.92,3.2,0.435,0.257,0.093,3.36,0.206,0.351,0.286,0.290,0.145,0.266,25.7,9.3,30.3,29.2,88.6,32.3,7.5,38.9,44.0,pitcher
6,592332,32,32,193.0,3.59,1.06,189,50,21,8.81,2.33,0.98,Kevin Gausman,P,Toronto Blue Jays,3.41,3.77,3.79,4.1,0.367,0.244,0.065,3.74,0.238,0.407,0.301,0.289,0.169,0.273,24.4,6.5,26.4,31.7,89.3,35.0,8.5,41.4,37.3,pitcher
7,668678,33,33,192.0,4.83,1.26,175,66,31,8.20,3.09,1.45,Zac Gallen,P,Arizona Diamondbacks,4.50,4.12,4.24,1.1,0.436,0.215,0.081,4.27,0.242,0.434,0.320,0.307,0.192,0.315,21.5,8.1,23.7,28.6,90.1,34.3,9.7,43.0,44.4,pitcher
8,664285,31,31,192.0,3.66,1.24,187,68,15,8.77,3.19,0.70,Framber Valdez,P,Houston Astros,3.37,3.34,3.66,4.0,0.586,0.233,0.085,3.79,0.241,0.370,0.303,0.312,0.129,0.295,23.3,8.5,26.6,29.8,90.8,28.1,6.7,46.3,59.4,pitcher
9,694973,32,32,187.2,1.97,0.95,216,42,11,10.38,2.02,0.53,Paul Skenes,P,Pittsburgh Pirates,2.36,3.03,3.10,6.5,0.443,0.295,0.057,2.65,0.202,0.321,0.255,0.255,0.119,0.247,29.5,5.7,30.1,30.5,87.6,29.4,5.8,40.1,44.6,pitcher


## 7. Combine into single master table

Stack hitters and pitchers into one DataFrame. Columns unique to one type will be NaN for the other (e.g. `xwOBA` is NaN for pitchers, `xERA` is NaN for hitters).

In [46]:
master = pd.concat([master_hit, master_pit], ignore_index=True)
print(f"Master table: {master.shape[0]} rows × {master.shape[1]} columns")
print(f"  Hitters:  {(master['player_type'] == 'hitter').sum()}")
print(f"  Pitchers: {(master['player_type'] == 'pitcher').sum()}")

# Reorder columns: identifiers first, then stats
id_cols = ["mlbam_id", "name", "position", "team", "player_type"]
stat_cols = [c for c in master.columns if c not in id_cols]
master = master[id_cols + stat_cols]

master.head(5)

Master table: 1193 rows × 66 columns
  Hitters:  537
  Pitchers: 656


,mlbam_id,name,position,team,player_type,PA,AVG,OBP,SLG,OPS,HR,RBI,BB,SO,BB_pct,K_pct,wOBA,BABIP,wRC_plus,fg_WAR_bat,GB_pct,FB_pct,LD_pct,xBA,xSLG,xwOBA,xOBP,xISO,avg_swing_speed,fast_swing_rate,blasts_contact,blasts_swing,squared_up_contact,squared_up_swing,avg_swing_length,swords,attack_angle,attack_direction,ideal_angle_rate,vertical_swing_path,exit_velocity_avg,launch_angle_avg,sweet_spot_percent,barrel_batted_rate,GS,G,IP,ERA,WHIP,K_per9,BB_per9,HR_per9,FIP,xFIP,SIERA,fg_WAR_pit,fg_GB_pct,fg_K_pct,fg_BB_pct,xERA,k_percent,bb_percent,whiff_percent,chase_percent,hard_hit_percent,groundballs_percent
0,596019,Francisco Lindor,SS,New York Mets,hitter,732.0,0.267,0.346,0.466,0.812,31,86.0,65,131,8.9,17.9,0.350,0.288,129.0,6.3,0.377,0.402,0.221,0.260,0.454,0.345,0.342,0.194,71.0,16.4,13.9,11.1,33.8,26.9,7.5,14.0,11.1,-1.6,51.1,33.8,90.5,15.1,34.6,8.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,646240,Rafael Devers,DH,San Francisco Giants,hitter,729.0,0.252,0.372,0.479,0.851,35,109.0,112,192,15.4,26.3,0.365,0.307,135.0,3.3,0.427,0.401,0.172,0.244,0.487,0.367,0.368,0.242,71.6,19.8,17.9,12.3,31.5,21.7,7.6,28.0,10.0,4.5,59.5,29.9,93.5,12.6,34.6,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,660271,Shohei Ohtani,TWP,Los Angeles Dodgers,hitter,727.0,0.282,0.392,0.622,1.014,55,102.0,109,187,15.0,25.7,0.418,0.315,172.0,7.5,0.392,0.439,0.169,0.274,0.649,0.425,0.387,0.375,75.8,61.9,26.0,17.8,36.4,25.0,7.9,14.0,15.0,-2.0,53.3,36.9,94.9,15.0,35.9,23.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,656941,Kyle Schwarber,LF,Philadelphia Phillies,hitter,724.0,0.240,0.365,0.563,0.928,56,132.0,108,197,14.9,27.2,0.391,0.253,152.0,4.9,0.341,0.480,0.179,0.251,0.584,0.403,0.374,0.333,77.3,77.1,24.7,17.5,32.2,22.8,7.5,51.0,14.6,-5.8,64.3,30.3,94.3,20.1,35.3,20.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,621566,Matt Olson,1B,Atlanta Braves,hitter,724.0,0.272,0.366,0.484,0.850,29,95.0,91,176,12.6,24.3,0.366,0.333,136.0,4.7,0.395,0.400,0.205,0.249,0.492,0.360,0.347,0.244,73.9,41.3,18.1,13.5,29.3,21.9,7.5,21.0,8.7,1.1,48.2,33.9,93.3,14.8,37.1,14.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Data Quality

In [48]:
# 8. Null rates per column, split by player type
print("=== Null Rates by Player Type ===\n")

hit_nulls = master[master["player_type"] == "hitter"].isnull().mean().round(4)
pit_nulls = master[master["player_type"] == "pitcher"].isnull().mean().round(4)

null_report = pd.DataFrame({"hitter_null_rate": hit_nulls, "pitcher_null_rate": pit_nulls})
# Only show columns with any nulls, or all if none
has_nulls = null_report[(null_report > 0).any(axis=1)]
if len(has_nulls) > 0:
    print("Columns WITH nulls:")
    print(has_nulls.to_string())
    
    no_nulls = null_report[(null_report == 0).all(axis=1)]
    print(f"\n{len(no_nulls)} columns have 0% nulls")
else:
    print("No nulls in any column!")

=== Null Rates by Player Type ===

Columns WITH nulls:
                     hitter_null_rate  pitcher_null_rate
PA                                0.0             1.0000
AVG                               0.0             1.0000
OBP                               0.0             1.0000
SLG                               0.0             1.0000
OPS                               0.0             1.0000
RBI                               0.0             1.0000
BB_pct                            0.0             1.0000
K_pct                             0.0             1.0000
BABIP                             0.0             1.0000
wRC_plus                          0.0             1.0000
fg_WAR_bat                        0.0             1.0000
GB_pct                            0.0             1.0000
FB_pct                            0.0             1.0000
LD_pct                            0.0             1.0000
avg_swing_speed                   0.0             1.0000
fast_swing_rate                  

## 9. Save master table

In [50]:
out_path = os.path.join(DATA, "master_players.parquet")
master.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
print(f"Saved: {out_path} — {size_kb:.1f} KB")
print(f"Final shape: {master.shape}")
print(f"Columns: {list(master.columns)}")

Saved: ../data/master_players.parquet — 163.9 KB
Final shape: (1193, 66)
Columns: ['mlbam_id', 'name', 'position', 'team', 'player_type', 'PA', 'AVG', 'OBP', 'SLG', 'OPS', 'HR', 'RBI', 'BB', 'SO', 'BB_pct', 'K_pct', 'wOBA', 'BABIP', 'wRC_plus', 'fg_WAR_bat', 'GB_pct', 'FB_pct', 'LD_pct', 'xBA', 'xSLG', 'xwOBA', 'xOBP', 'xISO', 'avg_swing_speed', 'fast_swing_rate', 'blasts_contact', 'blasts_swing', 'squared_up_contact', 'squared_up_swing', 'avg_swing_length', 'swords', 'attack_angle', 'attack_direction', 'ideal_angle_rate', 'vertical_swing_path', 'exit_velocity_avg', 'launch_angle_avg', 'sweet_spot_percent', 'barrel_batted_rate', 'GS', 'G', 'IP', 'ERA', 'WHIP', 'K_per9', 'BB_per9', 'HR_per9', 'FIP', 'xFIP', 'SIERA', 'fg_WAR_pit', 'fg_GB_pct', 'fg_K_pct', 'fg_BB_pct', 'xERA', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'hard_hit_percent', 'groundballs_percent']
